# Question 3 --- W&B compression dashboard

This Kaggle notebook uploads the completed QAT sweep to Weights & Biases (W&B), shows a local parallel-coordinates preview, and prepares the W&B dashboard chart for the report.

Before running, upload a Kaggle dataset containing either the extracted `Single_Precision_artifacts/` and `Mixed_Precision_artifacts/` directories, or their `*-artifacts.tgz` archives. For each run, keep `experiments/sweeps/<run>/metrics.json` and `results/logs/<run>-held-out-test.log` together under the same artifact directory. Store your W&B API key in Kaggle Secrets as `WANDB_API_KEY`.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# Set this to your Kaggle dataset slug after adding it through the right-side Input panel.
ARTIFACT_INPUT = '/kaggle/input/q3-compression-artifacts'
WANDB_PROJECT = 'cs6886-assignment2'
WANDB_ENTITY = None  # e.g. 'your-wandb-team'; leave None for your personal account

assert os.path.isdir(ARTIFACT_INPUT), f'Add the artifact dataset, then update ARTIFACT_INPUT: {ARTIFACT_INPUT}'
WANDB_API_KEY = UserSecretsClient().get_secret('WANDB_API_KEY')
assert WANDB_API_KEY, 'Create a Kaggle secret named WANDB_API_KEY.'


In [ ]:
# W&B and Plotly are used only for reporting; no model training occurs in this notebook.
%pip install -q 'wandb>=0.19,<1.0' 'plotly>=5.0' pandas

import wandb
wandb.login(key=WANDB_API_KEY)


In [ ]:
# If the Kaggle dataset contains .tgz archives, unpack each archive into an independent
# directory. Extracted artifact directories work too, so this cell is safe to run either way.
import tarfile
from pathlib import Path

input_root = Path(ARTIFACT_INPUT)
staging_root = Path('/kaggle/working/q3-artifacts')
staging_root.mkdir(parents=True, exist_ok=True)
archives = sorted(input_root.rglob('*.tgz')) + sorted(input_root.rglob('*.tar.gz'))
for archive in archives:
    destination = staging_root / archive.name.replace('.tar.gz', '').replace('.tgz', '')
    destination.mkdir(exist_ok=True)
    with tarfile.open(archive, 'r:gz') as tar:
        tar.extractall(destination)  # Archives were created locally by the QAT notebook.

# Search both the mounted dataset and extracted archives.
artifact_roots = [input_root, staging_root]
metrics_paths = sorted({path for root in artifact_roots for path in root.glob('**/experiments/sweeps/*/metrics.json')})
assert metrics_paths, 'No run metrics found. Check the uploaded folder structure.'
print(f'Found {len(metrics_paths)} run records:')
for path in metrics_paths:
    print(' -', path)


In [ ]:
import json
import re
import pandas as pd
import plotly.express as px

test_pattern = re.compile(r'Test top-1 accuracy:\s*([0-9.]+)%')
rows = []
for metrics_path in metrics_paths:
    metrics = json.loads(metrics_path.read_text())
    run_name = metrics['run_name']
    log_path = metrics_path.parents[3] / 'results' / 'logs' / f'{run_name}-held-out-test.log'
    match = test_pattern.search(log_path.read_text())
    assert match, f'Cannot parse held-out accuracy from {log_path}'
    nominal_w = metrics['weight_bits']
    realized = metrics.get('realized_weight_bits', {})
    rows.append({
        'run_name': run_name,
        'weight_bits': nominal_w,
        'activation_bits': metrics['activation_bits'],
        'weight_exception_layers': sum(bits != nominal_w for bits in realized.values()),
        'model_size_mib': metrics['total_bytes'] / 2**20,
        'weight_compression_ratio': metrics['weight_compression_ratio'],
        'activation_traffic_compression_ratio': metrics['activation_compression_ratio'],
        'validation_accuracy': metrics['best_target_validation_accuracy'],
        'test_accuracy': float(match.group(1)),
    })

results = pd.DataFrame(rows).sort_values(['test_accuracy', 'model_size_mib'], ascending=[False, True])
display(results.round(3))
axes = ['weight_bits', 'activation_bits', 'weight_exception_layers', 'model_size_mib',
        'weight_compression_ratio', 'activation_traffic_compression_ratio', 'test_accuracy']
fig = px.parallel_coordinates(results, dimensions=axes, color='test_accuracy',
                              color_continuous_scale=px.colors.sequential.Viridis,
                              title='Q3 compression sweep (local preview)')
fig.show()


In [ ]:
# Upload one W&B run per saved QAT run. Config fields and summary metrics become
# selectable axes in the W&B Parallel Coordinates panel. Re-running is safe: W&B
# creates a run with the same display name; use a fresh project if you want a clean dashboard.
for row in rows:
    config = {key: row[key] for key in ['weight_bits', 'activation_bits', 'weight_exception_layers']}
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, group='q3-compression',
                     name=row['run_name'], config=config, reinit=True)
    run.summary.update({key: row[key] for key in [
        'model_size_mib', 'weight_compression_ratio',
        'activation_traffic_compression_ratio', 'validation_accuracy', 'test_accuracy'
    ]})
    run.finish()

print(f'Uploaded {len(rows)} runs to https://wandb.ai/{WANDB_ENTITY or "<your-user>"}/{WANDB_PROJECT}')


## Create the report chart

Open the printed W&B project URL, then choose **Add panel** → **Parallel Coordinates**. Add these axes in order: `config.weight_bits`, `config.activation_bits`, `config.weight_exception_layers`, `summary.model_size_mib`, `summary.weight_compression_ratio`, `summary.activation_traffic_compression_ratio`, and `summary.test_accuracy`. Set the line color to `summary.test_accuracy`. Download the panel PNG as `wandb_parallel_coordinates.png`, put it beside `Q_3.tex`, and compile the report. The TeX source already includes this image automatically.